# 07 Testing and Code Structure

## Goal

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Goal</b><br>
Structure a Databricks project like software: transformation logic in <b>pure functions</b> inside modules, thin notebooks that call them, <b>unit tests</b> with pytest and PySpark's testing utilities (<code>assertDataFrameEqual</code>, <code>assertSchemaEqual</code>), data tests on real tables, and a project layout with Databricks Asset Bundles and CI that runs the tests before deploying.
</div>

## Why it matters

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Why it matters</b><br>
Untested pipelines break silently when someone changes a join or a filter. Teams that test their transformations catch bugs in a pull request instead of in a board meeting. CI/CD is 10% of the Data Engineer Associate exam, and "how do you test Spark code?" is a standard question for mid-level and senior roles.
</div>

## Concept

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 Testable pipeline code</b><br>
<code>`</code><br>my_pipeline/<br>├── databricks.yml              # bundle: targets (dev, prod), variables<br>├── resources/<br>│   └── orders_job.yml          # job definition (tasks, schedule, parameters)<br>├── src/<br>│   ├── my_pipeline/<br>│   │   ├── __init__.py<br>│   │   └── transforms.py       # pure functions: DataFrame in -> DataFrame out<br>│   └── notebooks/<br>│       └── build_orders.py     # thin: read tables, call transforms, write tables<br>└── tests/<br>    ├── conftest.py             # spark fixture<br>    └── test_transforms.py      # unit tests with small in-memory DataFrames<br><code>`</code><br><br>
| Test type | What it checks | Data | When |<br>|---|---|---|---|<br>| <b>Unit test</b> | One function's logic | Tiny DataFrames built in the test | Every commit (CI) |<br>| <b>Integration test</b> | A job or pipeline end to end | A dev or staging catalog | Before deploying to prod |<br>| <b>Data test</b> | Properties of real data (uniqueness, nulls, ranges) | Production tables | After each run (expectations, checks) |<br><br>
Keys to testability: functions take and return DataFrames, don't read or write tables themselves, don't use global state, and take parameters (dates, thresholds) as arguments.
</div>

## Real-world example

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🏦 Real-world example</b><br>
A refactoring changed a <code>left</code> join to an <code>inner</code> join in a customer metrics job, and 4% of customers vanished from a churn report for six weeks. Afterwards the team moved the logic into <code>transforms.py</code>, wrote tests such as "customers without orders keep a row with zero revenue", and made CI run pytest on every pull request with Databricks Connect. The next similar mistake failed in CI within two minutes of being pushed.
</div>

## Syntax

```python
from pyspark.testing import assertDataFrameEqual, assertSchemaEqual

def test_add_revenue(spark):
    df = spark.createDataFrame([(1, 2, 5.0)], "order_id INT, qty INT, price DOUBLE")
    expected = spark.createDataFrame([(1, 2, 5.0, 10.0)], "order_id INT, qty INT, price DOUBLE, revenue DOUBLE")
    assertDataFrameEqual(add_revenue(df), expected)
```

```bash
pytest tests/                        # locally or in CI, with Databricks Connect providing `spark`
databricks bundle validate           # check the bundle configuration
databricks bundle deploy -t dev      # deploy jobs and code to the dev target
databricks bundle run -t dev orders_job
```

## Setup

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Installs pytest if needed (on Databricks), and creates a folder for the project files: a workspace folder on Databricks, a temporary folder locally.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The project folder path.
</div>

In [0]:
%pip install -q pytest

In [0]:
CATALOG = "workspace"
DB = f"{CATALOG}.workflows"
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {DB}")
me = spark.sql("SELECT current_user()").first()[0]
PROJECT = f"/Workspace/Users/{me}/databricks-learning-workflows/testing_demo"

In [0]:
import os, sys, importlib
os.makedirs(f"{PROJECT}/tests", exist_ok=True)
sys.dont_write_bytecode = True                   # don't write __pycache__ folders into the workspace
print(PROJECT)

## 1. Pure transformation functions

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes <code>transforms.py</code> with three functions: clean orders, add revenue, and compute revenue per customer (keeping customers without orders).<br><br>
<b>Why it matters</b><br>Each function takes DataFrames and returns a DataFrame, and does nothing else: no reads, no writes, no <code>spark.table</code> calls. That makes them testable with tiny in-memory data, and reusable from notebooks, jobs and pipelines.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The module is imported, and a quick call shows revenue per customer, with customer 3 at 0.0.
</div>

In [0]:
TRANSFORMS = '''
from pyspark.sql import DataFrame, functions as F

def clean_orders(orders: DataFrame) -> DataFrame:
    """Keep orders with an id, a positive quantity and a non-negative price; deduplicate by order_id."""
    return (orders.filter(F.col("order_id").isNotNull() & (F.col("qty") > 0) & (F.col("price") >= 0))
                  .dropDuplicates(["order_id"]))

def add_revenue(orders: DataFrame) -> DataFrame:
    return orders.withColumn("revenue", F.col("qty") * F.col("price"))

def revenue_per_customer(customers: DataFrame, orders: DataFrame) -> DataFrame:
    """One row per customer, including customers without orders (revenue 0)."""
    per_customer = orders.groupBy("customer_id").agg(F.sum("revenue").alias("revenue"))
    return (customers.join(per_customer, "customer_id", "left")
                     .select("customer_id", F.coalesce("revenue", F.lit(0.0)).alias("revenue")))
'''
with open(f"{PROJECT}/transforms.py", "w") as f:
    f.write(TRANSFORMS)
if PROJECT not in sys.path:
    sys.path.insert(0, PROJECT)
import transforms
importlib.reload(transforms)

customers = spark.createDataFrame([(1,), (2,), (3,)], "customer_id INT")
orders = spark.createDataFrame([(10, 1, 2, 5.0), (11, 1, 1, 3.0), (12, 2, 4, 2.5), (12, 2, 4, 2.5), (13, 2, 0, 9.0)],
                               "order_id INT, customer_id INT, qty INT, price DOUBLE")
transforms.revenue_per_customer(customers, transforms.add_revenue(transforms.clean_orders(orders))).orderBy("customer_id").show()

## 2. Unit tests with PySpark's testing utilities

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Tests each function with small DataFrames, using <code>assertDataFrameEqual</code> (compares rows, ignoring order by default) and <code>assertSchemaEqual</code>.<br><br>
<b>Why it matters</b><br>These utilities give clear failure messages showing the differing rows, which is much better than comparing <code>collect()</code> results by hand.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All four checks pass and print <code>ok</code>.
</div>

In [0]:
from pyspark.testing import assertDataFrameEqual, assertSchemaEqual

def check_clean_orders_removes_bad_rows_and_duplicates(spark):
    got = transforms.clean_orders(orders)
    assertDataFrameEqual(got.select("order_id"), spark.createDataFrame([(10,), (11,), (12,)], "order_id INT"))

def check_add_revenue(spark):
    df = spark.createDataFrame([(1, 1, 2, 5.0)], "order_id INT, customer_id INT, qty INT, price DOUBLE")
    expected = spark.createDataFrame([(1, 1, 2, 5.0, 10.0)], "order_id INT, customer_id INT, qty INT, price DOUBLE, revenue DOUBLE")
    assertDataFrameEqual(transforms.add_revenue(df), expected)

def check_customers_without_orders_get_zero(spark):
    got = transforms.revenue_per_customer(customers, transforms.add_revenue(transforms.clean_orders(orders)))
    assertDataFrameEqual(got, spark.createDataFrame([(1, 13.0), (2, 10.0), (3, 0.0)], "customer_id INT, revenue DOUBLE"))

def check_output_schema(spark):
    got = transforms.revenue_per_customer(customers, transforms.add_revenue(orders))
    from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
    assertSchemaEqual(got.schema, StructType([StructField("customer_id", IntegerType()), StructField("revenue", DoubleType())]))

for check in [check_clean_orders_removes_bad_rows_and_duplicates, check_add_revenue, check_customers_without_orders_get_zero, check_output_schema]:
    check(spark)
    print(f"ok   {check.__name__}")

## 3. A failing test shows what's wrong

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Introduces the bug from the real-world example (an <code>inner</code> join instead of <code>left</code>) and runs the same test against it.<br><br>
<b>Why it matters</b><br>This is the point of tests: the failure message tells you exactly which rows differ, before the code ships.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The test fails, and the message shows the missing row for customer 3.
</div>

In [0]:
from pyspark.sql import functions as F

def buggy_revenue_per_customer(customers, orders):
    per_customer = orders.groupBy("customer_id").agg(F.sum("revenue").alias("revenue"))
    return customers.join(per_customer, "customer_id", "inner").select("customer_id", "revenue")

try:
    got = buggy_revenue_per_customer(customers, transforms.add_revenue(transforms.clean_orders(orders)))
    assertDataFrameEqual(got, spark.createDataFrame([(1, 13.0), (2, 10.0), (3, 0.0)], "customer_id INT, revenue DOUBLE"))
except AssertionError as e:
    print("TEST FAILED as it should:\n", str(e)[:600])

## 4. Running pytest from a notebook

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the tests as a pytest file <code>tests/test_transforms.py</code>, and runs pytest from the notebook, providing the existing <code>spark</code> session through a small plugin fixture.<br><br>
<b>Why it matters</b><br>The same test file runs in CI (where <code>spark</code> comes from Databricks Connect or a local session in <code>conftest.py</code>) and here in the notebook. One test suite, two places to run it.
</div>
<br>
<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>🆓 Free Edition note</b><br>
pytest is installed with <code>%pip</code> at the top of this notebook. The plugin passes the notebook's session to the tests, so they run on serverless compute.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
pytest reports <code>4 passed</code>, and the exit code is 0. <code>run_tests()</code> clears previously imported test modules first, so re-running after editing a test file picks up the changes.
</div>

In [0]:
TESTS = '''
from pyspark.testing import assertDataFrameEqual
import transforms

def make_orders(spark):
    return spark.createDataFrame([(10, 1, 2, 5.0), (11, 1, 1, 3.0), (12, 2, 4, 2.5), (12, 2, 4, 2.5), (13, 2, 0, 9.0)],
                                 "order_id INT, customer_id INT, qty INT, price DOUBLE")

def test_clean_orders_drops_bad_rows_and_duplicates(spark):
    got = transforms.clean_orders(make_orders(spark)).select("order_id")
    assertDataFrameEqual(got, spark.createDataFrame([(10,), (11,), (12,)], "order_id INT"))

def test_add_revenue(spark):
    df = spark.createDataFrame([(1, 1, 2, 5.0)], "order_id INT, customer_id INT, qty INT, price DOUBLE")
    assert transforms.add_revenue(df).first()["revenue"] == 10.0

def test_customers_without_orders_get_zero(spark):
    customers = spark.createDataFrame([(1,), (2,), (3,)], "customer_id INT")
    orders = transforms.add_revenue(transforms.clean_orders(make_orders(spark)))
    got = transforms.revenue_per_customer(customers, orders)
    assertDataFrameEqual(got, spark.createDataFrame([(1, 13.0), (2, 10.0), (3, 0.0)], "customer_id INT, revenue DOUBLE"))

def test_no_negative_revenue(spark):
    orders = transforms.add_revenue(transforms.clean_orders(make_orders(spark)))
    assert orders.filter("revenue < 0").count() == 0
'''
with open(f"{PROJECT}/tests/test_transforms.py", "w") as f:
    f.write(TESTS)

import pytest

class NotebookSpark:
    """pytest plugin that hands the notebook's Spark session to tests as the `spark` fixture."""
    @pytest.fixture
    def spark(self):
        return spark

def run_tests():
    for name in [m for m in sys.modules if m.startswith("test_")]:
        del sys.modules[name]                    # forget test modules imported by an earlier run, so edits are picked up
    return int(pytest.main([f"{PROJECT}/tests", "-q", "-p", "no:cacheprovider", "--rootdir", PROJECT], plugins=[NotebookSpark()]))

print("pytest exit code:", run_tests())

## 5. Data tests on real tables

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Writes the pipeline output to a table, then runs data tests on it: unique keys, no nulls in required columns, values in range, and a row count matching the input.<br><br>
<b>Why it matters</b><br>Unit tests check logic. Data tests check the data the logic produced, after every run, catching surprises in the input (a new duplicate source, a null key). They're the batch counterpart of pipeline expectations.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
All data checks pass.
</div>

In [0]:
out_table = f"{DB}.customer_revenue"
transforms.revenue_per_customer(customers, transforms.add_revenue(transforms.clean_orders(orders))).write.mode("overwrite").saveAsTable(out_table)
out = spark.table(out_table)

data_checks = {
    "customer_id unique": out.groupBy("customer_id").count().filter("count > 1").count() == 0,
    "no null customer_id": out.filter("customer_id IS NULL").count() == 0,
    "revenue non-negative": out.filter("revenue < 0").count() == 0,
    "one row per customer": out.count() == customers.count(),
}
for name, passed in data_checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {name}")
assert all(data_checks.values()), "data tests failed"

## 6. The project as a bundle, tested in CI

<div style="background:#E8F1FF;border-left:5px solid #2F6FED;padding:10px 14px;border-radius:6px;color:#1b2a41;">
<b>💡 What it does</b><br>
Prints the two configuration files that turn this folder into a deployable project: the bundle definition with dev and prod targets, and a GitHub Actions workflow that runs the tests and deploys.<br><br>
<b>Why it matters</b><br>This is the production workflow: code and job definitions live in Git, every pull request runs the tests, and merges deploy the same bundle to prod with different variables (catalog, schedule). It's covered hands-on in <code>09_azure_databricks</code>.
</div>
<br>
<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Expected output</b><br>
The YAML for <code>databricks.yml</code> and the CI workflow.
</div>

In [0]:
print('''# databricks.yml
bundle:
  name: orders_pipeline
include:
  - resources/*.yml
variables:
  catalog:
    default: dev
targets:
  dev:
    mode: development          # prefixes resources with your name, pauses schedules
    default: true
  prod:
    mode: production
    variables:
      catalog: prod
''')
print('''# .github/workflows/ci.yml
name: ci
on: [pull_request, push]
jobs:
  test-and-deploy:
    runs-on: ubuntu-latest
    env:
      DATABRICKS_HOST: ${{ secrets.DATABRICKS_HOST }}
      DATABRICKS_CLIENT_ID: ${{ secrets.DATABRICKS_CLIENT_ID }}          # service principal (OAuth M2M)
      DATABRICKS_CLIENT_SECRET: ${{ secrets.DATABRICKS_CLIENT_SECRET }}
    steps:
      - uses: actions/checkout@v4
      - uses: databricks/setup-cli@main
      - run: pip install pytest databricks-connect
      - run: pytest tests/
      - run: databricks bundle validate
      - if: github.ref == 'refs/heads/main'
        run: databricks bundle deploy -t prod
''')

## Under the hood

```
pull request ─▶ CI: pytest (unit tests, tiny DataFrames via Databricks Connect / local Spark)
             ─▶ databricks bundle validate
merge to main ─▶ databricks bundle deploy -t prod ─▶ jobs + code updated in the prod workspace
job run ─▶ notebook (thin) ─▶ transforms.py (tested) ─▶ tables ─▶ data tests / expectations
```

<span style="background:#D64545;color:white;padding:2px 10px;border-radius:12px;font-size:12px;">Action</span> unit tests run actions on tiny DataFrames, so the whole suite takes seconds to minutes.

In [0]:
print(sorted(os.listdir(PROJECT)), sorted(os.listdir(f"{PROJECT}/tests")))

## Common mistakes and troubleshooting log

<div style="background:#FDEAEA;border-left:5px solid #D64545;padding:10px 14px;border-radius:6px;color:#4a1414;">
<b>⛔ Problem: logic that reads and writes tables inside the function</b><br>
Untestable without real tables. Pass DataFrames in, return DataFrames out, and keep I/O in the notebook.<br><br>
**⛔ Problem: tests that compare <code>collect()</code> lists in a fixed order**<br>Row order isn't guaranteed. Use <code>assertDataFrameEqual</code> (order-insensitive by default).<br><br>
<b>⛔ Problem: tests using production data</b><br>Slow, flaky and a privacy risk. Build small DataFrames in the test, including edge cases (nulls, duplicates, empty input).<br><br>
**⛔ Problem: <code>ModuleNotFoundError</code> in tests**<br>The module's folder isn't on <code>sys.path</code>. Use a package layout with <code>src/</code>, a <code>conftest.py</code>, or run pytest with <code>--rootdir</code>.<br><br>
<b>⛔ Problem: changes to a module not picked up in the notebook</b><br>Use <code>importlib.reload</code> or restart Python.<br><br>
<b>✍️ My own problems</b><br>(add yours here)
</div>

## Interview Q&A

<div style="background:#F1E9FF;border-left:5px solid #7A4DD8;padding:10px 14px;border-radius:6px;color:#2d1a57;">
<b>🎤 1. How do you unit-test PySpark code?</b><br>
I keep transformations as pure functions that take and return DataFrames, then write pytest tests that build tiny DataFrames with edge cases, call the function, and compare with <code>assertDataFrameEqual</code> and <code>assertSchemaEqual</code>. A <code>spark</code> fixture comes from a local session or Databricks Connect in CI.<br><br>

<b>🎤 2. What's the difference between unit tests and data tests?</b><br>
Unit tests check code logic with constructed inputs, before deployment. Data tests (or expectations) check properties of real data after each run: unique keys, no nulls in required columns, valid ranges, expected row counts.<br><br>

<b>🎤 3. How would you structure a Databricks project in Git?</b><br>
A bundle (<code>databricks.yml</code> with dev and prod targets), job and pipeline definitions in <code>resources/</code>, logic in a Python package under <code>src/</code>, thin notebooks or entry points that do I/O and call the package, and <code>tests/</code> with pytest. CI runs tests and <code>bundle validate</code> on pull requests and deploys on merge.<br><br>

<b>🎤 4. How do you run tests against Databricks from CI?</b><br>
With Databricks Connect, which gives the tests a Spark session backed by Databricks compute, authenticated as a service principal (OAuth). Alternatively, run tests as a job task after deploying to a dev target.
</div>

## Certification corner

<div style="background:#E6F6F7;border-left:5px solid #0E8C8C;padding:10px 14px;border-radius:6px;color:#0b3536;">
<b>🎓 Exam-style questions (Data Engineer Associate)</b><br><br>
<b>Q1. Which design makes a transformation easiest to unit-test?</b><br>
A. A notebook that reads a table, transforms it and writes the result in one cell<br>
B. A function that takes a DataFrame and returns a DataFrame, with I/O done elsewhere<br>
C. A <code>%run</code> notebook that defines global variables<br>
D. A SQL query stored in a dashboard<br>
<details><summary><b>Show answer</b></summary><b>B.</b></details><br><br>

<b>Q2. Which tool packages job definitions and code for deployment to dev and prod workspaces from Git?</b><br>
A. Databricks Asset Bundles<br>
B. <code>dbutils.notebook.run</code><br>
C. Delta Sharing<br>
D. Unity Catalog volumes<br>
<details><summary><b>Show answer</b></summary><b>A.</b></details><br><br>

<b>Q3. Which function compares two DataFrames in a test, ignoring row order by default?</b><br>
A. <code>df1 == df2</code><br>
B. <code>pyspark.testing.assertDataFrameEqual</code><br>
C. <code>df1.exceptAll(df2)</code> only<br>
D. <code>assert df1.count() == df2.count()</code><br>
<details><summary><b>Show answer</b></summary><b>B.</b></details>
</div>

## Practice problem

<div style="background:#FFEFE5;border-left:5px solid #F07D2E;padding:10px 14px;border-radius:6px;color:#4a2300;">
<b>🧪 Without copying from above:</b>
<ol><li>Add a function <code>top_customers(revenue_df, n)</code> to <code>transforms.py</code> that returns the <code>n</code> customers with the highest revenue (ties broken by <code>customer_id</code>), and write a pytest test for it, including <code>n</code> larger than the number of customers</li><li>Add a test that <code>clean_orders</code> returns an empty DataFrame (with the same schema) for an empty input</li><li>Run pytest again and confirm all tests pass</li></ol>
</div>

In [0]:
# Your solution for the practice problem goes here.
# Add a comment above each step explaining what it does.

## Solution

<div style="background:#FFF4DB;border-left:5px solid #E5A100;padding:10px 14px;border-radius:6px;color:#4a3500;">
<b>⚠️ Try it yourself first</b><br>
Only look at this after you have a working answer of your own.
</div>

In [0]:
# 1. New function
with open(f"{PROJECT}/transforms.py", "a") as f:
    f.write('''
def top_customers(revenue: DataFrame, n: int) -> DataFrame:
    return revenue.orderBy(F.desc("revenue"), F.asc("customer_id")).limit(n)
''')

# 1 + 2. New tests
with open(f"{PROJECT}/tests/test_transforms.py", "a") as f:
    f.write('''
def test_top_customers(spark):
    rev = spark.createDataFrame([(1, 5.0), (2, 9.0), (3, 9.0)], "customer_id INT, revenue DOUBLE")
    assert [r.customer_id for r in transforms.top_customers(rev, 2).collect()] == [2, 3]
    assert transforms.top_customers(rev, 10).count() == 3

def test_clean_orders_empty_input(spark):
    empty = spark.createDataFrame([], "order_id INT, customer_id INT, qty INT, price DOUBLE")
    got = transforms.clean_orders(empty)
    assert got.count() == 0 and got.schema == empty.schema
''')

# 3. Run the suite (reload the module so pytest sees the new function)
importlib.reload(transforms)
print("pytest exit code:", run_tests())

## Summary and cheat sheet

<div style="background:#E7F7EE;border-left:5px solid #1E9E5A;padding:10px 14px;border-radius:6px;color:#14391f;">
<b>✅ Key takeaways</b>
<ul><li>Pure functions (DataFrame in, DataFrame out) in modules, thin notebooks for I/O</li><li>Unit tests with tiny DataFrames and <code>assertDataFrameEqual</code> / <code>assertSchemaEqual</code></li><li>Data tests or expectations on real outputs after each run</li><li>One pytest suite runs in the notebook (via a fixture plugin) and in CI (Databricks Connect)</li><li>Bundles + CI: test on pull requests, deploy on merge</li></ul>
</div>

| Need | Use |
|---|---|
| Compare DataFrames | `assertDataFrameEqual(actual, expected)` |
| Compare schemas | `assertSchemaEqual(a.schema, b.schema)` |
| Run tests in a notebook | `pytest.main([path, "-p", "no:cacheprovider"], plugins=[...])` (clear cached `test_*` modules first) |
| Spark in CI | Databricks Connect + `conftest.py` fixture |
| Deploy | `databricks bundle deploy -t <target>` |

## Git commit

```
git add 07_workflows/07_testing_and_code_structure.ipynb
git commit -m "add 07_07 testing and code structure notebook"
```